# Main-paper mechanism plots (Figures 5–7)

This notebook redraws three main-paper figures from a compact numeric archive.
The plot functions follow `local_ideb/iclr_review/vis.ipynb`.
The file loading and output paths are adapted for this repository.
The original scientific source files are unchanged.

Run all cells from the repository root or from `plots/`.
The normal run needs no private source files and runs no ID estimation experiments.
It writes the three PDF plots to the ignored `plots/generated/` folder.

The archive contains only plotted values. It uses lossless NPZ compression.
Figure 5 keeps the stored means and standard errors for MNIST and MLE.
Figure 6 keeps all 50 anchor spectra at the eight plotted neighborhood sizes.
Figure 7 keeps the histogram values and skewness. These values were regenerated
from the original saved samples with the original notebook functions and seed.
The manifest records every source file, timestamp, hash and selection.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

# This notebook can run with the repository root or plots/ as the working directory.
ROOT = Path.cwd() if (Path.cwd() / "data" / "mechanism_data.npz").is_file() else Path.cwd().parent
DATA = ROOT / "data"
if not (DATA / "mechanism_data.npz").is_file():
    raise FileNotFoundError("Run this notebook from the repository root or plots/.")
OUT = ROOT / "plots" / "generated"
OUT.mkdir(parents=True, exist_ok=True)
with np.load(DATA / "mechanism_data.npz", allow_pickle=False) as archive:
    summary = {key: archive[key] for key in archive.files}
manifest = json.loads((DATA / "mechanism_manifest.json").read_text())
print("Loaded", len(summary), "numeric arrays.")


## Optional: rebuild the compressed archive

Skip the next cell for normal plot reproduction. It defines an author export function.
To rebuild, set `ORIGINAL_RESULTS_ROOT` to the local `local_ideb` directory.
The function reads that directory and writes only to this repository's `data/` directory.
It selects the Figure 5 and Figure 6 values. It also reruns the original Figure 7
histogram calculation on 2,000 points per dataset. It does not rerun ID estimation.
The original files are read only. Full cosine arrays are temporary and are discarded.

The four sampling and histogram functions below are copied from the original notebook.
The exporter converts their outputs to numeric arrays and records provenance.


In [ ]:

from datetime import datetime, timezone
import json
import hashlib
import pickle
import platform
import h5py
import scipy
import sklearn
from scipy.linalg import qr
from scipy.stats import skew
from sklearn.neighbors import NearestNeighbors as NN

# Parameters from the original notebook, cells 36 and 43.
MAX_SAMPLE = 10_000
di, da, seed, N, k, n_jobs = 12, 20, 42, 2000, 100, 10

def hSphere_isometry(di=3, da=6, N=MAX_SAMPLE, seed= 42):
    np.random.seed(seed)
    # Wr = np.random.randn(di,da)
    Wr, _ = qr(np.random.randn(da, di+1), mode='economic')
    Wr = Wr.T
    samples = np.random.randn(N,di+1)
    samples = samples/np.linalg.norm(samples, axis=1, keepdims=True)
    samples = samples @ Wr

    return samples

def abids_cosines(samples,knn,):
    N, dim = samples.shape
    dists, indices = knn
    cvs = []
    for i in range(N):
        neighbors = samples[indices[i]]-samples[i]
	    
        # Normalize neighbors and discard neighbors at distance 0
        neighbor_norms = np.linalg.norm(neighbors,axis=1)
        neighbors = neighbors[neighbor_norms > 0] / neighbor_norms[neighbor_norms > 0][:,None]
        
        # For the formula we need the neighborhood size
        k = neighbors.shape[0]
        
        # For unit vectors, cosine is just the dot product #
        cosines = neighbors.dot(neighbors.T)
        # exclude the diagonals here #
        mask = ~np.eye(cosines.shape[0], dtype=bool)
        cosines_no_diag = cosines[mask].reshape(cosines.shape[0], -1)

        cvs.append(cosines_no_diag)
        # print(cosines.shape)
        # cvs.append(cosines)
    
    cvs = np.concatenate(cvs, axis=0)
    cvs = np.reshape(cvs, newshape=-1)

    return cvs

def abid_interp_pipeline(sp_unpert,di,info,eps=0.0,
                         k=k,seed=seed,num=100):
    np.random.seed(seed)
    if eps > 0.0:
        distort = np.random.rand(N,1)*eps + 1 - eps/2.0 
    else:
        distort = np.ones((N,1))
    samples = distort*sp_unpert

    nbr = NN(n_neighbors=k,algorithm='auto',n_jobs=n_jobs).fit(samples)
    knn = nbr.kneighbors(samples)

    cvs = abids_cosines(samples=samples, knn=knn)

    n = float(cvs.shape[0])

    m = cvs.mean()
    exp_var = 1/di
    sk=skew(cvs)

    bins = np.linspace(-1,1,num=num)
    bins_idx = 0.5*(bins[1:]+bins[:-1])
    # print(bins.shape) 

    # Compute histogram : #
    hist,_ = np.histogram(cvs, bins=bins,density=False)
    hist = np.float_(hist)/n

    # hist /= n
    
    info[f'eps:{eps:.2f}'] = (hist, m, sk, exp_var,eps, bins_idx)

def abid_interp_pipeline_quest(samples,di,info_quest,
                               sample_name:str,num=100):
    nbr = NN(n_neighbors=k,algorithm='auto',n_jobs=n_jobs).fit(samples)
    knn = nbr.kneighbors(samples)

    cvs = abids_cosines(samples=samples, knn=knn)

    n = float(cvs.shape[0])

    m = cvs.mean()
    sk = skew(cvs)
    exp_var = 1/di

    bins = np.linspace(-1,1,num=num)
    bins_idx = 0.5*(bins[1:]+bins[:-1])
    # print(bins.shape) 

    # Compute histogram : #
    hist,_ = np.histogram(cvs, bins=bins,density=False)
    hist = np.float_(hist)/n
    # hist /= n
    
    info_quest[sample_name] = (hist, m, sk, exp_var, bins_idx)


def export_mechanism_data(source_root, destination=DATA):
    """Read original result files. Save only the numbers used by Figures 5–7."""
    source_root = Path(source_root).resolve()
    origin = source_root / "iclr_review"
    destination = Path(destination).resolve()
    if destination == source_root or destination.is_relative_to(source_root):
        raise ValueError("The export destination must be outside the original source directory.")
    arrays = {}
    records = []

    def record(relative_path, fields, figure, action):
        path = source_root / relative_path
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        stat = path.stat()
        records.append({
            "path": relative_path,
            "size_bytes": stat.st_size,
            "mtime_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
            "sha256": digest.hexdigest(),
            "fields_used": fields,
            "figure": figure,
            "action": action,
        })

    def load_pickle(name):
        with (origin / "data" / name).open("rb") as stream:
            return pickle.load(stream)

    squeeze = load_pickle("squeeze_mnist_mle.pkl")
    noise = load_pickle("noise_mnist_mle.pkl")
    classes = np.asarray(squeeze["classes"])
    eps_values = np.asarray(squeeze["eps_list"])
    sigma_values = np.asarray(noise["sigma2_list"])
    cov_types = ["iso", "aniso", "corr"]
    arrays["mnist_classes"] = classes
    arrays["mnist_eps"] = eps_values
    arrays["mnist_sigma2"] = sigma_values
    for stat in ["mean", "stderr"]:
        arrays["mnist_squeeze_" + stat] = np.asarray([
            [squeeze[c][eps][stat] for eps in eps_values] for c in classes
        ])
        arrays["mnist_noise_" + stat] = np.asarray([
            [[noise[c][ct][s2][stat] for s2 in sigma_values] for ct in cov_types]
            for c in classes
        ])
    record("iclr_review/data/squeeze_mnist_mle.pkl", ["classes", "eps_list", "[class][epsilon].mean", "[class][epsilon].stderr"], 5, "Exact selection; pointwise lides discarded. Stored summary values are unchanged.")
    record("iclr_review/data/noise_mnist_mle.pkl", ["classes", "sigma2_list", "[class][covariance][variance].mean", "[class][covariance][variance].stderr"], 5, "Exact selection; pointwise lides discarded. Covariance order: iso, aniso, corr.")

    spectra = [
        ("sphere_clean", "sphere_25_51_0.0__spectral_gap.pkl"),
        ("sphere_distorted", "sphere_25_51_1.998__spectral_gap.pkl"),
        ("grassmann", "gr_proj_10_6_spectral_gap.pkl"),
    ]
    spectral_selection = {}
    for label, name in spectra:
        result = load_pickle(name)
        grid = np.asarray(result["k_values"])
        selected_indices = np.round(np.linspace(0, len(grid)-1, 8)).astype(int)
        chosen = grid[selected_indices]
        anchors = np.asarray(result["anchor_idx"])
        arrays[label + "_k"] = chosen
        arrays[label + "_anchors"] = anchors
        for position, kval in enumerate(chosen):
            arrays[label + f"_evals_{position}"] = np.stack([
                np.asarray(result["evals"][idx][kval]) for idx in anchors
            ])
        spectral_selection[label] = {"original_k_grid_length": len(grid), "selected_grid_indices": selected_indices.tolist(), "k_values": chosen.tolist(), "anchors": len(anchors), "N": result["N"], "di": result["di"], "da": result["da"]}
        record("iclr_review/data/" + name, ["k_values", "anchor_idx", "evals[anchor][k]"], 6, "Lossless selection of the eight k values used by vis.ipynb cell 30. Every plotted anchor and eigenvalue is retained.")

    # The original notebook stores these histograms only in memory.
    # Recompute them with its unchanged functions and source samples.
    sp_unpert = hSphere_isometry(di=di, da=da, N=N, seed=seed)
    eps_vals = [0.0, 0.5, 1.0, 1.5, 2.0]
    info = {}
    for eps in eps_vals:
        abid_interp_pipeline(sp_unpert=sp_unpert, di=di, info=info, eps=eps, seed=seed)
        print("ABID sphere epsilon", eps, flush=True)
    arrays["abid_eps"] = np.asarray(eps_vals)
    arrays["abid_sphere_hist"] = np.stack([info[f"eps:{eps:.2f}"][0] for eps in eps_vals])
    arrays["abid_sphere_skew"] = np.asarray([info[f"eps:{eps:.2f}"][2] for eps in eps_vals])
    arrays["abid_bins"] = info["eps:0.00"][5]
    sample_paths = {
        "gman_proj": "_gman_proj_p_(10, 6, 42).h5",
        "gman_vec": "_gman_vec_p_(10, 5, 42).h5",
        "st_proj": "_steifel_proj_p_(10, 6, 42).h5",
        "st_vec1": "_steifel_vec1_p_(9, 5, 42).h5",
        "flag_vec": "_flag_vec_p_(6, 3, 2, 42).h5",
    }
    info_quest = {}
    for label, name in sample_paths.items():
        with h5py.File(origin / "data" / "samples" / name, "r") as stream:
            samples = stream["samples"][:N]
        samples = samples.reshape(N, -1)
        abid_interp_pipeline_quest(samples=samples, di=di, info_quest=info_quest, sample_name=label)
        values = info_quest[label]
        arrays[label + "_abid_hist"] = values[0]
        arrays[label + "_abid_skew"] = np.asarray(values[2])
        record("iclr_review/data/samples/" + name, ["samples[:2000]"], 7, "Histogram and skewness regenerated using the original vis.ipynb functions (cells 39, 46, 49). Full pairwise cosine arrays are discarded after aggregation.")
        print("ABID HomID", label, flush=True)
    record("iclr_review/vis.ipynb", ["cells 5, 9 (Figure 5)", "cells 20, 30 (Figure 6)", "cells 36, 39, 43, 44, 46–50, 52 (Figure 7)"], [5, 6, 7], "Original plotting and histogram functions copied. Only file loading, output paths and notebook organization are adapted. Pauli is excluded because the Figure 7 plotting cell explicitly skips it.")
    destination.mkdir(parents=True, exist_ok=True)
    archive_path = destination / "mechanism_data.npz"
    np.savez_compressed(archive_path, **arrays)
    # Check that compression preserves every selected numeric array.
    with np.load(archive_path, allow_pickle=False) as saved:
        for key, value in arrays.items():
            np.testing.assert_array_equal(saved[key], value)
    manifest = {
        "schema_version": 1,
        "archive": "mechanism_data.npz",
        "compression": "NumPy NPZ / ZIP DEFLATE, lossless; original numeric precision retained",
        "source_root": "../local_ideb (relative to the repository root; not needed for plotting)",
        "source_notebook": "iclr_review/vis.ipynb",
        "figure_mapping": {"5": "mle_squeeze_noise.pdf", "6": "lpca_interp_mt.pdf", "7": "abid_interp.pdf"},
        "figure_6_selection": spectral_selection,
        "figure_7_derivation": {
            "status": "Regenerated from original source inputs, not a preexisting numerical cache",
            "sphere_di": di, "sphere_da": da, "N": N, "k_including_self": k, "seed": seed, "n_jobs": n_jobs,
            "eps_values": eps_vals, "histogram_edges": "np.linspace(-1, 1, num=100)",
            "histogram_normalization": "Counts divided by the total number of off-diagonal cosine values; 99 bins.",
            "sphere_generation": "Original hSphere_isometry (QR random isometry and normalized Gaussian samples). Radial distortion: eps*U(0,1) + 1 - eps/2, reseeded with 42 for each epsilon.",
            "homid_selection": "First 2000 saved samples per manifold, flattened before kNN.",
            "reference_curve": "Original cell 52 beta-function expression, normalized by its discrete sum. Kept unchanged.",
        },
        "environment": {"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__, "scikit_learn": sklearn.__version__, "h5py": h5py.__version__},
        "arrays": {key: {"shape": list(value.shape), "dtype": str(value.dtype)} for key, value in arrays.items()},
        "archive_size_bytes": archive_path.stat().st_size,
        "archive_sha256": hashlib.sha256(archive_path.read_bytes()).hexdigest(),
        "sources": records,
    }
    (destination / "mechanism_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
    return manifest

# Optional author export. The public archive is sufficient for all plots below.
# Set a path only if the original result directory is available locally.
ORIGINAL_RESULTS_ROOT = None
if ORIGINAL_RESULTS_ROOT is not None:
    export_mechanism_data(ORIGINAL_RESULTS_ROOT)


## Figure 5: MNIST MLE under distortion and noise

Each color is one MNIST class. Shaded bands show the stored standard error.
The right panel uses a logarithmic noise-variance axis.
The mean values and bands are copied directly from the saved result files.


In [ ]:
METHODS = ["mle"]
cov_types = ["iso", "aniso", "corr"]
linestyles = {"iso": "-", "aniso": "--", "corr": ":"}

def load_method(method_name):
    if method_name != "mle":
        raise ValueError("The main-paper archive contains the MLE panel only.")
    classes = summary["mnist_classes"]
    eps_values, sigma_values = summary["mnist_eps"], summary["mnist_sigma2"]
    squeeze = {"classes": classes, "eps_list": eps_values}
    noise = {"classes": classes, "sigma2_list": sigma_values}
    for i, cls in enumerate(classes):
        squeeze[cls] = {
            eps: {stat: summary["mnist_squeeze_" + stat][i, j] for stat in ["mean", "stderr"]}
            for j, eps in enumerate(eps_values)
        }
        noise[cls] = {
            ct: {s2: {stat: summary["mnist_noise_" + stat][i, ct_i, j] for stat in ["mean", "stderr"]}
                 for j, s2 in enumerate(sigma_values)}
            for ct_i, ct in enumerate(cov_types)
        }
    return squeeze, noise

def plot_single_method(method_name, save=True):
    results_squeeze, results_noise = load_method(method_name)

    classes = results_squeeze['classes']
    eps_list = results_squeeze['eps_list']
    sigma2_list = results_noise['sigma2_list']
    colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))

    fig, axs = plt.subplots(1, 2, figsize=(18, 5), sharex=False, sharey=False)

    # ------ Left: Squeezing ------
    for i, c in enumerate(classes):
        means = [results_squeeze[c][eps]['mean'] for eps in eps_list]
        stes  = [results_squeeze[c][eps]['stderr'] for eps in eps_list]

        axs[0].plot(eps_list, means, color=colors[i], marker='.')
        axs[0].fill_between(eps_list,
                            np.array(means) - np.array(stes),
                            np.array(means) + np.array(stes),
                            color=colors[i], alpha=0.2)

    axs[0].set_xlabel("$\\epsilon$",fontsize=20)
    axs[0].set_ylabel("$\hat{d_i}$",fontsize=20)
    axs[0].set_title(f"Anisotropic Distortion",fontsize=20)
    axs[0].tick_params(axis='both', which='major', labelsize=18)
    axs[0].grid(True)
    

    # ------ Right: Noise ------
    for i, c in enumerate(classes):
        for ct in cov_types:
            means = [results_noise[c][ct][s2]['mean'] for s2 in sigma2_list]
            stes  = [results_noise[c][ct][s2]['stderr'] for s2 in sigma2_list]

            axs[1].plot(sigma2_list, means,
                        linestyle=linestyles[ct],
                        marker='.',
                        color=colors[i])
            axs[1].fill_between(sigma2_list,
                                np.array(means) - np.array(stes),
                                np.array(means) + np.array(stes),
                                color=colors[i], alpha=0.15)

    axs[1].set_xscale('log')
    axs[1].set_xlabel("$\\sigma^2$",fontsize=20)
    # axs[1].set_ylabel("ID Est",fontsize=20)
    axs[1].set_title(f"Additive Noise", fontsize=20)
    axs[1].tick_params(axis='both', which='major', labelsize=18)
    axs[1].grid(True)


    # ------ Legend ------
    class_handles = [mpatches.Patch(color=colors[i], label=f"Class {c}")
                     for i, c in enumerate(classes)]
    style_handles = [
        mlines.Line2D([], [], color='k', linestyle='-', label='Isotropic'),
        mlines.Line2D([], [], color='k', linestyle='--', label='Anisotropic'),
        mlines.Line2D([], [], color='k', linestyle=':', label='Correlated')
    ]

    leg1 = axs[1].legend(handles=class_handles, title="MNIST Classes",
                         loc='upper left', bbox_to_anchor=(1.02, 1.0),fontsize=14)
    leg2 = axs[1].legend(handles=style_handles, title="Covariance Type",
                         loc='lower left', bbox_to_anchor=(1.02, 0.0),fontsize=14)
    axs[1].add_artist(leg1)

    # fig.suptitle(f"{METHOD_NAMES[method_name]}", fontsize=16)
    fig.tight_layout(rect=[0, 0, 0.9, 1])

    # ---- SAVE ----
    if save:
        fig.savefig(OUT / f"{method_name}_squeeze_noise.pdf",
                    dpi=300, bbox_inches="tight")

    plt.show()

plot_single_method("mle", save=True)


## Figure 6: local PCA spectra

The panels show an undistorted sphere, a distorted sphere, and a Grassmannian projector embedding.
Every faint curve is one saved anchor. Colors select the eight original neighborhood sizes.
The component-index axis is logarithmic. The vertical line marks the dimension used in the original figure.
The archive preserves the original component indexing and plotting conventions.


In [ ]:
SAMPLES = {"gman_proj": (None, "gr_proj_10_6")}

def load_sg_results(fn_prefix):
    label = {
        "sphere_25_51_0.0_": "sphere_clean",
        "sphere_25_51_1.998_": "sphere_distorted",
        "gr_proj_10_6": "grassmann",
    }[fn_prefix]
    anchors, kvals = summary[label + "_anchors"], summary[label + "_k"]
    evals = {
        idx: {kval: summary[label + f"_evals_{j}"][i] for j, kval in enumerate(kvals)}
        for i, idx in enumerate(anchors)
    }
    return {"anchor_idx": anchors, "k_values": kvals, "evals": evals}

from matplotlib.lines import Line2D

def plot_rel_evals_single(ax, evals_dict, k_values_all, anchor_idx, title, di, xlabel=False, ylabel=False):
    """
    Plot only the relative eigenvalues curves in a given axis.
    """
    chosen_k_indices = np.round(np.linspace(0, len(k_values_all)-1, 8)).astype(int)
    chosen_k_values  = k_values_all[chosen_k_indices]
    colors = [f"C{i}" for i in range(len(chosen_k_values))]

    for ci, k in enumerate(chosen_k_values):
        # stack evals across anchors
        eval_list = [np.array(evals_dict[idx][k]) for idx in anchor_idx]
        eval_k = np.stack(eval_list, axis=0)

        # rel_ev = np.cumsum(eval_k, axis=1)
        # rel_ev /= rel_ev[:, -1][:, None]

        for row in eval_k:
            ax.plot(np.arange(len(row)), row, color=colors[ci], alpha=0.2)

    ax.set_title(title, fontsize=32)
    if xlabel:
        ax.set_xlabel("Component index", fontsize=26)
    if ylabel:
        ax.set_ylabel("Relative Eigenvalues from PCA", fontsize=26)
    ax.tick_params('both', labelsize=24)
    ax.set_xscale("log")
    ax.grid(True)
    ax.set_ylim(-0.01,0.3)
    ax.axvline(di, ls='--', color='black')

    # legend handles:
    N = 10000
    legend_handles = [Line2D([0],[0], color=colors[i], lw=4, alpha=0.7)
                      for i in range(len(chosen_k_values))]
    legend_labels = [f"k/N={k/N:.2e}" for k in chosen_k_values]

    return legend_handles, legend_labels

def lpca_interp_mt():
    fig, axs = plt.subplots(1, 3, figsize=(24, 8), sharey = True, sharex= True)



    # ---------- (1) Sphere di=25, eps=0.0 ----------
    di = 25
    da = 2*di + 1
    fn_prefix = f"sphere_{di}_{da}_0.0_"
    data2 = load_sg_results(fn_prefix)
    handles2, labels2 = plot_rel_evals_single(
        axs[0], data2["evals"], np.array(data2["k_values"]),
        data2["anchor_idx"], r"Sphere, (Undistorted)", di =di, ylabel=True,
    )

    # ---------- (2) Sphere di=25, eps=1.998 ----------
    eps = 1.998
    fn_prefix = f"sphere_{di}_{da}_{eps}_"
    data3 = load_sg_results(fn_prefix)
    handles3, labels3 = plot_rel_evals_single(
        axs[1], data3["evals"], np.array(data3["k_values"]),
        data3["anchor_idx"], rf"Sphere (Distorted)", di =di, xlabel=True,

        
    )

    # ---------- (3)ax.set_xticks(ticksize=16) Gr(Proj) ----------
    di = 24
    (_, fn_prefix) = SAMPLES["gman_proj"]
    data = load_sg_results(fn_prefix)
    handles1, labels1 = plot_rel_evals_single(
        axs[2], data["evals"], np.array(data["k_values"]),
        data["anchor_idx"], "Gr (Proj)", di =di, ylabel=False,
    )

    # Combine legends (all ks identical, so reuse the first)
    fig.legend(handles1, labels1, loc="upper right", fontsize=14)

    fig.tight_layout()
    fig.savefig(OUT / "lpca_interp_mt.pdf", dpi=300, bbox_inches="tight")
    print("Saved lpca_interp_mt.pdf")
    plt.show()

lpca_interp_mt()


## Figure 7: neighborhood-angle histograms

The left panel compares five radial distortions of a sphere.
The right panel compares the undistorted sphere with the five plotted HomID embeddings.
The original code uses 2,000 points and 100 nearest neighbors, including the query point.
It discards zero-distance vectors and diagonal cosine entries.
Each histogram contains 99 bins. Bin counts are divided by the total cosine count.
The reference curve and legend skewness follow the original notebook.


In [ ]:
from scipy.special import beta

di = 12
eps_vals = summary["abid_eps"]
bins_idx = summary["abid_bins"]
info = {
    f"eps:{eps:.2f}": (
        summary["abid_sphere_hist"][i], None,
        summary["abid_sphere_skew"][i], 1/di, eps, bins_idx,
    )
    for i, eps in enumerate(eps_vals)
}
MANIFOLDS = ["gman_proj", "gman_vec", "st_proj", "st_vec1", "flag_vec"]
MANIFOLD_NAMES = {
    "gman_proj": "Gr (Proj)", "gman_vec": "Gr (Vec)",
    "st_proj": "St (Matrix)", "st_vec1": "St (Vec)",
    "flag_vec": "Flag (Vec)", "sphere": "Sphere",
}
info_quest = {
    mn: (summary[mn + "_abid_hist"], None,
         summary[mn + "_abid_skew"], 1/di, bins_idx)
    for mn in MANIFOLDS
}
hist, mean, skewness, exp_var, _, bins_idx = info["eps:0.00"]
info_quest["sphere"] = hist, mean, skewness, exp_var, bins_idx



from scipy.special import beta 
a = b = 0.5*(di-1)
x = 0.5*(1+bins_idx)
nab = beta(a,b)
th = 0.5*(x**a)*((1-x)**b)/nab
th/= th.sum()


XLIM=(-0.25,0.95)

fig, axs = plt.subplots(1,2,figsize=(24,8),sharex=True, sharey=False)

# Plot for distorted sphere #
ax = axs[0]

for i, eps, in enumerate(eps_vals):
    hist, m, sk, exp_var, eps, bins_idx = info[f'eps:{eps:.2f}']
    
    ax.plot(bins_idx, hist, c=f'C{i+1}', label=f"$\\epsilon : ${eps:.2f}, $g_1$ : {sk:.2f}",lw=4)
    # ax.axvspan(m-exp_var, m+exp_var, color=f'C{i+1}',alpha=0.3)

ax.plot(bins_idx, th, c = 'C0',ls = '-.', label='Theory')

ax.grid(True)
ax.set_xlabel('cos $\\theta$',fontsize=18)
ax.set_ylabel('P(cos $\\theta$)',fontsize=18)
ax.legend(fontsize=12)
ax.tick_params(axis='both', which='major', labelsize=18)
ax.set_xlim(XLIM)
ax.set_title('Distorted spheres',fontsize=18)

# Plot for different QUEST manifolds #
ax=axs[1]
MANIFOLD_TO_PLOT = ['sphere'] + MANIFOLDS

for i, mn in enumerate(MANIFOLD_TO_PLOT):
    if 'pauli' in mn:
        continue
    hist, m, sk, exp_var, bins_idx = info_quest[mn]
    
    ax.plot(bins_idx, hist, c=f'C{i+1}', label=f"{MANIFOLD_NAMES[mn]}, $g_1$ : {sk:.2f}", lw=4)
    # ax.axvspan(m-exp_var, m+exp_var, color=f'C{i+1}',alpha=0.3)

ax.plot(bins_idx, th, c = 'C0',ls = '-.', label='Theory')

ax.grid(True)
ax.set_xlabel('cos $\\theta$',fontsize=18)
# ax.set_ylabel('P(cos $\\theta$)',fontsize=18)
ax.legend(fontsize=12)
ax.set_xlim(XLIM)
ax.set_title('HomID manifolds',fontsize=18) 

ax.tick_params(axis='both', which='major', labelsize=18)

fn = OUT / "abid_interp.pdf"
fig.savefig(fn, format='pdf', bbox_inches='tight')
plt.show()


In [ ]:
# Confirm that all expected outputs exist and all selected curves are finite.
for filename in ["mle_squeeze_noise.pdf", "lpca_interp_mt.pdf", "abid_interp.pdf"]:
    path = OUT / filename
    assert path.exists() and path.stat().st_size > 0, path
    print(path.name, path.stat().st_size, "bytes")
for key in summary:
    assert np.isfinite(summary[key]).all(), key
for key in ["abid_sphere_hist"] + [mn + "_abid_hist" for mn in MANIFOLDS]:
    np.testing.assert_allclose(summary[key].sum(axis=-1), 1, atol=1e-6)
print("Output and numeric checks passed.")
